# Model Evaluation & Quality Gates

In CD4ML, models are guarded by statistical quality gates before promotion.
We score a **labeled holdout** through the same inference path production
uses — a SageMaker transform job on the inference image — then compare
predictions against ground truth.

For churn the headline metric is **AUC** (does the model rank churners
above stayers?) plus the confusion matrix at a chosen operating threshold:
the campaign favors recall over precision, per the problem framing.


## Environment & local SageMaker session

In [ ]:
import os

# Kernel env on PATH so %%bash cells can call feast/aws/docker directly; the
# aws CLI and Feast binaries live in the workspace venv (/opt/mlops-venv).
os.environ["PATH"] = "/opt/mlops-venv/bin:" + os.environ["PATH"]
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")

# This project's own moto (see docker-compose.yml) — NOT the lab's shared one.
os.environ["MOTO_URL"] = "http://telco-customer-churn-moto:5000"

In [ ]:
import os
from pathlib import Path

import yaml
from sagemaker_local.config import LocalModeConfig
from sagemaker_local.patches import cleanup_stale_serving_containers
from sagemaker_local.session import make_local_session

# A killed run can leave a serving container bound to this project's host
# port; drop leftovers before we start (idempotent).
cleanup_stale_serving_containers()

# SAGEMAKER_LOCAL_REPO_PATH must resolve identically on the host and inside
# this container — local mode bind-mounts it into job/serving containers.
PROJECT_DIR = (
    Path(os.environ["SAGEMAKER_LOCAL_REPO_PATH"])
    / "projects"
    / "telco_customer_churn"
)

cfg = LocalModeConfig(
    **yaml.safe_load((PROJECT_DIR / "configs" / "local.yaml").read_text())
)
boto_session, sm_session = make_local_session(cfg)

TRAIN_IMAGE = "telco-customer-churn:train"
INFERENCE_IMAGE = "telco-customer-churn:inference"
print("s3 endpoint:", cfg.s3_endpoint_url, "| serving port:", cfg.serving_port)

## Train a candidate

Evaluation needs an artifact; the fit is the same Estimator call the
training stage walks through.


In [ ]:
import pandas as pd
from feast import FeatureStore
from sagemaker.estimator import Estimator

fs = FeatureStore(repo_path="../feature_repo")
labels = pd.read_parquet("../data/processed/churn_labels.parquet")
training_df = fs.get_historical_features(
    entity_df=labels,
    features=[
        f"customer_subscription:{name}"
        for name in [
            "senior_citizen",
            "partner",
            "dependents",
            "tenure_months",
            "paperless_billing",
            "month_to_month",
            "one_year_contract",
            "electronic_check",
            "dsl",
            "fiber_optic",
            "streaming_tv",
            "streaming_movies",
            "num_add_ons",
            "monthly_charges",
            "total_charges",
        ]
    ],
).to_df()
training_df.to_csv("../data/processed/train.csv", index=False)

In [ ]:
%%bash
aws s3 cp ../data/processed/train.csv     "s3://telco-customer-churn/input/train/train.csv"     --endpoint-url "$MOTO_URL"

In [ ]:
est = Estimator(
    entry_point="train.py",
    source_dir=str(PROJECT_DIR / "src" / "telco_customer_churn"),
    image_uri=TRAIN_IMAGE,
    role=cfg.role_arn,
    instance_type="local",
    instance_count=1,
    sagemaker_session=sm_session,
    output_path=f"s3://{cfg.bucket}/models",
    hyperparameters={"n_estimators": 100, "max_depth": 4},
)
est.fit({"train": f"s3://{cfg.bucket}/input/train/"})
print(est.model_data)

## Build the evaluation input

The transform input is the *labelled* population — features in the contract
order, headerless (the serving input parser reads a raw matrix).


In [ ]:
import json

import numpy as np

schema = json.loads(
    (PROJECT_DIR / "data" / "processed" / "feature_schema.json").read_text()
)
eval_df = fs.get_historical_features(
    entity_df=labels.sample(frac=0.3, random_state=7),
    features=[f"customer_subscription:{name}" for name in schema["features"]],
).to_df()

input_dir = PROJECT_DIR / "data" / "input"
input_dir.mkdir(parents=True, exist_ok=True)
input_csv = input_dir / "eval.csv"
np.savetxt(
    input_csv,
    eval_df[schema["features"]].astype("float64").to_numpy(),
    delimiter=",",
)
len(eval_df)

## Score the holdout through the inference image

`Model(...).transformer(...)` runs the batch-transform job: a serving
container on the inference image loads `model.joblib` and streams the input.


In [ ]:
import shutil

from sagemaker.model import Model

output_dir = PROJECT_DIR / "data" / "output"
shutil.rmtree(output_dir, ignore_errors=True)

transformer = Model(
    image_uri=INFERENCE_IMAGE,
    model_data=est.model_data,
    role=cfg.role_arn,
    sagemaker_session=sm_session,
    name="telco-churn-eval-model",
).transformer(
    instance_count=1,
    instance_type="local",
    output_path=f"file:{output_dir}",
)
transformer.transform(
    f"file://{input_dir}",
    split_type="Line",
    content_type="text/csv",
)

## Confusion matrix & AUC


In [ ]:
predictions = json.loads((output_dir / "eval.csv.out").read_text())
eval_df = eval_df.assign(prediction=predictions)

threshold = 0.5
tp = int(
    ((eval_df["prediction"] >= threshold) & (eval_df["churn"] == 1)).sum()
)
fp = int(
    ((eval_df["prediction"] >= threshold) & (eval_df["churn"] == 0)).sum()
)
fn = int(((eval_df["prediction"] < threshold) & (eval_df["churn"] == 1)).sum())
tn = int(((eval_df["prediction"] < threshold) & (eval_df["churn"] == 0)).sum())
recall = tp / max(tp + fn, 1)
precision = tp / max(tp + fp, 1)
print(f"threshold={threshold}  TP={tp} FP={fp} FN={fn} TN={tn}")
print(f"recall={recall:.3f}  precision={precision:.3f}")

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

auc = roc_auc_score(eval_df["churn"], eval_df["prediction"])
fpr, tpr, _ = roc_curve(eval_df["churn"], eval_df["prediction"])


plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f"AUC = {auc:.3f}")
plt.plot([0, 1], [0, 1], "k--", alpha=0.4)
plt.xlabel("false positive rate")
plt.ylabel("true positive rate (recall)")
plt.legend()
plt.title("ROC — holdout scored via batch transform")
plt.show()

import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

## Quality gate

A sensible promotion gate for this dataset: **AUC ≥ 0.80** — below that the
ranked list gives the campaign no real lift. The assert below is the gate
as code; a pipeline would block promotion when it fails.


In [ ]:
assert auc >= 0.80, f"model under the promotion gate: AUC={auc:.3f} < 0.80"
print(f"quality gate passed: AUC={auc:.3f} >= 0.80")